# Task 05A — resumable A100 campaign

Select **Runtime → Change runtime type → A100 GPU**. Google Drive stores resumable checkpoints only; the notebook never pushes to GitHub. The first session profiles `trpb_seed0` and stops. Later sessions run every missing shard automatically and download one final ZIP after all 20 shards aggregate successfully. Task 05B is not included.


In [ ]:
REPO_URL = 'https://github.com/PaulsonLab/energy-inference-bo.git'
REPO_REF = 'main'  # A full 40-character commit SHA is preferred after publication.
RUN_PROFILE = False   # First scientific session only: runs trpb_seed0, then stops.
RUN_CAMPAIGN = False  # Later sessions: resumes every remaining shard.
SESSION_BUDGET_HOURS = 8
SHARD_TIMEOUT_SECONDS = 2 * 60 * 60 + 55 * 60
assert not (RUN_PROFILE and RUN_CAMPAIGN), 'Choose profile or campaign, never both.'


## Setup and frozen preflight

This creates the locked project environment without restarting Colab, records the resolved Git SHA, verifies the A100, runs the unit suite, and runs the non-scientific smoke wiring check. Rerunning at the same revision is safe.


In [ ]:
import json, os, pathlib, re, subprocess, sys
repo = pathlib.Path('/content/energy-inference-bo-task05a')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
if not repo.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(repo)], check=True)
assert (repo / '.git').exists(), f'{repo} exists but is not the Task 05A clone'
assert not subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain', '--untracked-files=no'], text=True).strip(), 'tracked Colab clone changes would be overwritten'
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', '--prune'], check=True)
target = REPO_REF if re.fullmatch(r'[0-9a-fA-F]{40}', REPO_REF) else f'origin/{REPO_REF}'
subprocess.run(['git', '-C', str(repo), 'cat-file', '-e', f'{target}^{{commit}}'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', target], check=True)
sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
print('Pinned Git SHA:', sha)
env = os.environ.copy(); env['MPLBACKEND'] = 'Agg'; env['PYTHONUNBUFFERED'] = '1'
subprocess.run(['uv', 'sync', '--locked', '--group', 'dev', '--no-editable'], cwd=repo, env=env, check=True)
py = repo / '.venv/bin/python'
probe = subprocess.run([str(py), '-c', "import json,sys,torch,botorch,gpytorch,numpy,scipy; print(json.dumps({'python':sys.version,'torch':torch.__version__,'botorch':botorch.__version__,'gpytorch':gpytorch.__version__,'numpy':numpy.__version__,'scipy':scipy.__version__,'cuda':torch.cuda.is_available(),'device':torch.cuda.get_device_name() if torch.cuda.is_available() else None}))"], cwd=repo, env=env, text=True, capture_output=True, check=True)
runtime = json.loads(probe.stdout); print(json.dumps(runtime, indent=2))
assert runtime['cuda'] and 'A100' in runtime['device'], f"A100 required, got {runtime['device']}"
subprocess.run([str(py), '-m', 'pytest', '-q'], cwd=repo, env=env, check=True)
smoke_dir = repo / 'artifacts/task05a/colab_smoke' / sha
subprocess.run([str(py), '-m', 'energy_bo.experiments.run_task05a', '--profile', 'smoke', '--device', 'cuda', '--output-dir', str(smoke_dir)], cwd=repo, env=env, check=True)
smoke_gate = json.loads((smoke_dir / 'gate_result.json').read_text())
assert smoke_gate['status'] == 'INCONCLUSIVE' and smoke_gate['gate_checks'].get('smoke_only') is True
print('Smoke wiring gate:', smoke_gate['status'], '(cannot pass Task 05A)')


## Persistent campaign status

Authorize the Drive mount. Evidence is stored under `MyDrive/energy-inference-bo/task05a/<git-sha>/`. This cell prints all 20 shards, their validated progress, and the exact next shard. A stale `RUNNING` shard becomes resumable `PARTIAL` after a disconnect.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
campaign_root = pathlib.Path('/content/drive/MyDrive/energy-inference-bo/task05a') / sha
campaign_root.mkdir(parents=True, exist_ok=True)
base_cmd = [str(py), '-m', 'energy_bo.experiments.run_task05a_campaign', '--campaign-root', str(campaign_root), '--repo-dir', str(repo), '--device', 'cuda']
def run_control(arguments):
    completed = subprocess.run(arguments, cwd=repo, env=env, text=True, capture_output=True)
    if completed.returncode:
        print(completed.stdout); print(completed.stderr)
        completed.check_returncode()
    return json.loads(completed.stdout)
run_control([*base_cmd, '--mode', 'status'])
print((campaign_root / 'CAMPAIGN_STATUS.md').read_text())


## First session: profile exactly one shard

Set `RUN_PROFILE=True` in the configuration cell, rerun from the top, and run this cell. It runs or resumes only `trpb_seed0`, records its actual A100 duration, prints the updated status, and intentionally stops. Do not enable `RUN_CAMPAIGN` in the profiling session.


In [ ]:
if not RUN_PROFILE:
    print('Profile guard active. Set RUN_PROFILE=True for the first scientific session only.')
else:
    profile_result = run_control([*base_cmd, '--mode', 'profile', '--shard-timeout-seconds', str(SHARD_TIMEOUT_SECONDS)])
    print((campaign_root / 'CAMPAIGN_STATUS.md').read_text())
    print('Seed-0 technical profile (not a scientific gate):')
    print(json.dumps(profile_result['profile_review'], indent=2))
    print('Profile session complete. Review trpb_seed0 runtime, then use RUN_CAMPAIGN=True in a later session.')


## Later sessions: resume the complete campaign

After the profile succeeds, set `RUN_PROFILE=False` and `RUN_CAMPAIGN=True`, then run the notebook from the top. This cell skips complete shards and resumes partial ones. It stops before starting a shard unlikely to fit inside the eight-hour soft budget. Reopen and repeat after a disconnect or clean budget stop. When all shards validate, it automatically aggregates them and downloads one final ZIP.


In [ ]:
if not RUN_CAMPAIGN:
    print('Campaign guard active. Set RUN_CAMPAIGN=True only after the profile session succeeds.')
else:
    run_control([*base_cmd, '--mode', 'campaign', '--session-budget-seconds', str(int(SESSION_BUDGET_HOURS * 3600)), '--shard-timeout-seconds', str(SHARD_TIMEOUT_SECONDS)])
    status = json.loads((campaign_root / 'campaign_status.json').read_text())
    print((campaign_root / 'CAMPAIGN_STATUS.md').read_text())
    failed = status['counts']['FAILED'] + status['counts']['INCOMPATIBLE']
    if failed:
        run_control([*base_cmd, '--mode', 'diagnostic'])
        print('Campaign stopped on a failed/incompatible shard. Diagnostic ZIP:', campaign_root / 'task05a_diagnostic.zip')
    elif status['counts']['COMPLETE'] < 20:
        print('Clean session stop. Reopen this notebook and run again with RUN_CAMPAIGN=True; completed work will be skipped.')
    else:
        zip_path = campaign_root / 'task05a_full_results.zip'
        assert zip_path.exists(), 'All shards completed but the final package is missing.'
        print('Full campaign and frozen aggregation complete:', zip_path)
        from google.colab import files
        files.download(str(zip_path))


## After download

Extract `task05a_full_results.zip` locally without rearranging it. The top-level `task05a/` folder contains `full_shards/`, `aggregate/`, campaign status files, the Colab manifest, and `SHA256SUMS.json`. Provide that extracted folder to the repository agent for checksum audit, frozen-gate review, compact evidence import, and tracking updates. Do not manually mark Task 05A PASS or begin Task 05B.
